# Notebook 03 — Modelagem Preditiva de Risco: Violência contra a Mulher (PI5)

## Contexto e Motivação

Este notebook implementa um pipeline de **Machine Learning** para prever o risco de ocorrência de agressões contra a mulher em Sorocaba e Votorantim, utilizando dados da Secretaria de Segurança Pública (DDM — Delegacia de Defesa da Mulher).

O objetivo não é substituir investigações policiais, mas **apoiar a alocação preventiva de recursos**, identificando bairros e períodos com maior probabilidade de ocorrência de violência doméstica.

---

## Estratégia Adotada: Amostragem Negativa Espaço-Temporal

Em vez de trabalhar apenas com os registros de ocorrências reais, expandimos o problema para o **universo completo de possibilidades**: toda combinação de bairro × mês × ano. Combinações sem ocorrência registrada recebem `ocorreu = 0`; combinações com ao menos uma ocorrência recebem `ocorreu = 1`.

Essa abordagem é chamada de **amostragem negativa** e permite que o modelo aprenda não apenas "onde o crime aconteceu", mas também "onde ele poderia ter acontecido e não aconteceu", tornando a predição muito mais rica.

---

## Pipeline do Notebook

| Etapa | Descrição |
|---|---|
| 1. Configuração | Instalação de pacotes e conexão ao BigQuery |
| 2. Preparação | Limpeza, diagnóstico de perda e engenharia de features |
| 3. Análise Exploratória | Visualizações de distribuição, sazonalidade e heatmap |
| 4. Divisão e Balanceamento | Split treino/teste com Target Encoding e Undersampling |
| 5. Função de Avaliação | Métricas com otimização de threshold |
| 6. Modelos Baseline | DummyClassifier e Regressão Logística |
| 7. Árvore de Decisão | Modelo interpretável de referência |
| 8. Random Forest | Ensemble de árvores com alta capacidade preditiva |
| 9. XGBoost | Modelo principal com busca de hiperparâmetros |
| 10. Comparação | Quadro e gráfico comparativo de todos os modelos |
| 11. Importância | Quais variáveis mais influenciam a previsão |
| 12. Predição | Exemplo prático de uso do modelo |
| 13. Exportação | Salvamento do modelo e artefatos para o aplicativo |


---
## 1. Configuração do Ambiente

Instalamos as bibliotecas necessárias e configuramos os parâmetros globais de visualização.

- **google-cloud-bigquery / pandas-gbq:** acesso ao banco de dados na nuvem
- **scikit-learn:** algoritmos de ML, pré-processamento e métricas
- **xgboost:** o algoritmo principal de gradient boosting
- **imbalanced-learn:** técnicas para lidar com dados desbalanceados (SMOTE, UnderSampler)


In [ ]:
!pip install google-cloud-bigquery pandas-gbq scikit-learn xgboost matplotlib seaborn imbalanced-learn -q

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from google.cloud import bigquery
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Ambiente configurado com sucesso.')

### 1.1 Conexão com o BigQuery

Os dados brutos das ocorrências já foram coletados, limpos e carregados no BigQuery pelo notebook `01_etl.ipynb`. Aqui apenas autenticamos e consultamos as colunas necessárias para a modelagem.

> **Alternativa offline:** se não houver acesso ao BigQuery, descomente a célula com `pd.read_csv` e aponte para o arquivo exportado manualmente.


In [ ]:
CREDENCIAIS_PATH = '../credenciais.json'
NOME_DO_PROJETO  = 'projetointegrador5'
ID_DA_TABELA     = 'dados_ssp.dados_ddm'

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = CREDENCIAIS_PATH
client = bigquery.Client(project=NOME_DO_PROJETO)

print(f'Conectado ao projeto: {NOME_DO_PROJETO}')
print(f'Tabela de origem    : {ID_DA_TABELA}')

In [ ]:
# Selecionamos apenas as colunas que farão parte do modelo.
QUERY = f"""
SELECT
    nome_municipio,
    ano_ocorrencia,
    mes_ocorrencia,
    periodo_ocorrencia,
    dia_semana,
    local_ocorrencia,
    bairro
FROM `{NOME_DO_PROJETO}.{ID_DA_TABELA}`
WHERE ano_ocorrencia >= 2022
ORDER BY ano_ocorrencia, mes_ocorrencia
"""

df_raw = client.query(QUERY).to_dataframe()

print(f'Registros carregados : {len(df_raw)}')
print(f'Período              : {df_raw["ano_ocorrencia"].min()} a {df_raw["ano_ocorrencia"].max()}')
print(f'Municípios           : {sorted(df_raw["nome_municipio"].dropna().unique().tolist())}')
df_raw.head(3)

In [ ]:
# Alternativa sem conexão ao BigQuery:
# Exportar a tabela como CSV pelo console do BigQuery e ajustar o caminho abaixo.

# df_raw = pd.read_csv('../data/processed/dados_ddm.csv', encoding='utf-8')
# print(f'{len(df_raw)} registros carregados do CSV')

---
## 2. Preparação dos Dados e Feature Engineering

### 2.1 Diagnóstico de Qualidade dos Dados

Antes de limpar ou remover qualquer registro, auditamos **qual campo é responsável pela maior perda de dados**. Isso é essencial para documentar o impacto de cada decisão de limpeza e justificar as escolhas metodológicas.

A análise responde: *o campo `bairro` ou o campo `periodo_ocorrencia` descarta mais registros? Existe sobreposição?*


In [ ]:
sem_info = ['Nao Informado', 'Não Informado', 'nan', 'None', '']
total = len(df_raw)

# ── Máscaras individuais ──────────────────────────────────────────────
mask_bairro       = df_raw['bairro'].astype(str).isin(sem_info)
mask_periodo_ni   = df_raw['periodo_ocorrencia'].astype(str).isin(sem_info)
mask_hora_incerta = df_raw['periodo_ocorrencia'].astype(str) == 'Em Hora Incerta'

perdidos_bairro       = mask_bairro.sum()
perdidos_periodo_ni   = mask_periodo_ni.sum()
perdidos_hora_incerta = mask_hora_incerta.sum()
perdidos_ambos        = (mask_bairro & mask_periodo_ni).sum()
so_bairro             = (mask_bairro  & ~mask_periodo_ni).sum()
so_periodo            = (~mask_bairro &  mask_periodo_ni).sum()

# ── Simula a cadeia EXATA da célula 'limpeza' ────────────────────────
df_s1 = df_raw[~df_raw['bairro'].astype(str).isin(sem_info)].copy()
df_s2 = df_s1[~df_s1['periodo_ocorrencia'].astype(str).isin(sem_info)].copy()
df_s3 = df_s2.dropna(subset=['bairro', 'periodo_ocorrencia', 'mes_ocorrencia'])

hora_incerta_em_s2 = df_s2['periodo_ocorrencia'].astype(str).eq('Em Hora Incerta').sum()

# ── Resultados ───────────────────────────────────────────────────────
print('=' * 58)
print(f'  DIAGNÓSTICO DE PERDA DE DADOS  —  {total} registros')
print('=' * 58)
print()
print('  FILTROS ISOLADOS (se apenas aquele filtro existisse):')
print(f'    bairro  = "Não Informado"    : {perdidos_bairro:>5}  ({perdidos_bairro/total:.1%})')
print(f'    período = "Não Informado"    : {perdidos_periodo_ni:>5}  ({perdidos_periodo_ni/total:.1%})')
print(f'    período = "Em Hora Incerta"  : {perdidos_hora_incerta:>5}  ({perdidos_hora_incerta/total:.1%})')
print()
print('  SOBREPOSIÇÃO:')
print(f'    Ambos inválidos ao mesmo tempo : {perdidos_ambos:>5}  ({perdidos_ambos/total:.1%})')
print()
print('  CAUSA EXCLUSIVA:')
print(f'    Somente o bairro é inválido    : {so_bairro:>5}  ({so_bairro/total:.1%})')
print(f'    Somente o período é inválido   : {so_periodo:>5}  ({so_periodo/total:.1%})')
print()
print('  FUNIL (replica exatamente a célula "limpeza"):')
print(f'    Bruto (BigQuery)               : {total:>5}')
print(f'    Após filtro bairro             : {len(df_s1):>5}  (−{total - len(df_s1)})')
print(f'    Após filtro período "Não Info" : {len(df_s2):>5}  (−{len(df_s1) - len(df_s2)})')
print(f'    "Em Hora Incerta" (passo 3)    : {hora_incerta_em_s2:>5}  → NaN no mapa → caem no dropna')
print(f'    Após dropna()                  : {len(df_s3):>5}  (−{len(df_s2) - len(df_s3)})')
print(f'    ─────────────────────────────────────────────────')
print(f'    TOTAL removido                 : {total - len(df_s3):>5}  ({(total - len(df_s3))/total:.1%})')
print(f'    TOTAL mantido                  : {len(df_s3):>5}  ({len(df_s3)/total:.1%})')
print()
print('  VALORES ÚNICOS em periodo_ocorrencia:')
for val, cnt in df_raw['periodo_ocorrencia'].value_counts(dropna=False).items():
    in_ni = str(val) in set(sem_info)
    in_hi = str(val) == 'Em Hora Incerta'
    if in_ni:
        marca = '✖ filtro "Não Informado"'
    elif in_hi:
        marca = '△ vira NaN no mapa → cai no dropna'
    else:
        marca = '✔ mantido'
    print(f'    {str(val):<22} {cnt:>5}  ({cnt/total:.1%})  {marca}')
print()
print('  VALORES inválidos em bairro:')
inv_bairro = df_raw[mask_bairro]['bairro'].value_counts(dropna=False)
if inv_bairro.empty:
    print('    (nenhum)')
else:
    for val, cnt in inv_bairro.items():
        print(f'    {str(val):<22} {cnt:>5}  ({cnt/total:.1%})')

In [ ]:
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Diagnóstico de Perda de Dados — DDM Sorocaba/Votorantim',
             fontsize=13, fontweight='bold', y=1.02)

# ── Gráfico 1: Funil ─────────────────────────────────────────────────
etapas = ['Bruto\n(BigQuery)', 'Após filtro\nbairro', 'Após filtro\nperíodo "NI"', 'Após\ndropna()']
valores = [total, len(df_s1), len(df_s2), len(df_s3)]
cores_f = ['#4C72B0', '#DD8452', '#FF7F0E', '#C44E52']

bars = axes[0].bar(etapas, valores, color=cores_f, width=0.5,
                   edgecolor='white', linewidth=1.2)
axes[0].set_title('Funil de Perda de Dados', fontsize=12, fontweight='bold', pad=10)
axes[0].set_ylabel('Registros')
axes[0].set_ylim(0, max(valores) * 1.22)

for bar, val in zip(bars, valores):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + max(valores) * 0.012,
                 f'{val:,}', ha='center', va='bottom',
                 fontsize=10, fontweight='bold')

for i in range(1, len(valores)):
    diff = valores[i - 1] - valores[i]
    if diff > 0:
        mid = (valores[i - 1] + valores[i]) / 2
        axes[0].annotate(f'−{diff:,}', xy=(i - 0.5, mid),
                         fontsize=9, color='#C44E52',
                         ha='center', fontweight='bold')

# ── Gráfico 2: Distribuição de periodo_ocorrencia ────────────────────
vc = df_raw['periodo_ocorrencia'].value_counts(dropna=False)

cores_p = []
for idx in vc.index:
    if str(idx) in set(sem_info):
        cores_p.append('#C44E52')    # vermelho — removido direto
    elif str(idx) == 'Em Hora Incerta':
        cores_p.append('#FF7F0E')    # laranja — cai no dropna
    else:
        cores_p.append('#55A868')    # verde — mantido

bars2 = axes[1].barh([str(x) for x in vc.index], vc.values,
                     color=cores_p, edgecolor='white', linewidth=1.2)
axes[1].set_title('Distribuição: periodo_ocorrencia', fontsize=12, fontweight='bold', pad=10)
axes[1].set_xlabel('Registros')
axes[1].set_xlim(0, vc.max() * 1.35)

for bar, val in zip(bars2, vc.values):
    axes[1].text(val + vc.max() * 0.01,
                 bar.get_y() + bar.get_height() / 2,
                 f'{val} ({val/total:.0%})',
                 va='center', fontsize=9)

legenda = [
    mpatches.Patch(facecolor='#55A868', label='✔ Mantido no modelo'),
    mpatches.Patch(facecolor='#FF7F0E', label='△ Em Hora Incerta → cai no dropna'),
    mpatches.Patch(facecolor='#C44E52', label='✖ Removido ("Não Informado")'),
]
axes[1].legend(handles=legenda, loc='lower right', fontsize=8.5)

plt.tight_layout()
plt.show()

print()
print(f'Conclusão: o maior responsável pela perda é o campo "periodo_ocorrencia"')
print(f'  {perdidos_periodo_ni} removidos por "Não Informado"')
print(f'  {hora_incerta_em_s2} removidos por "Em Hora Incerta" (vira NaN no mapa)')
print(f'  {so_bairro} removidos exclusivamente pelo bairro')

### 2.2 Limpeza e Tratamento de Valores Inválidos

Após o diagnóstico, aplicamos a estratégia de tratamento:

- **Bairros sem informação** (`"Não Informado"`, `"nan"` etc.) são agrupados na categoria `"Desconhecido"` em vez de serem descartados. Isso preserva o máximo de registros e garante que o modelo ainda aprenda sobre ocorrências sem geolocalização de bairro.
- **Registros sem `mes_ocorrencia`** são removidos (são raros e causados por falha de sistema — sem o mês, a feature temporal é inviável).
- A variável `periodo_ocorrencia` com valor `"Em Hora Incerta"` é mapeada como `-1` e eliminada pelo `dropna()` posteriormente, pois seu valor não contribui para aprendizado temporal.


In [ ]:
df = df_raw.copy()

# Em vez de jogar fora, agrupamos valores sem informação do bairro na categoria "Desconhecido"
sem_info = ['Nao Informado', 'Não Informado', 'nan', 'None', '']
df['bairro'] = df['bairro'].replace(sem_info, 'Desconhecido')

# Se alguma linha realmente não tiver o mês_ocorrencia por erro do sistema (raro), removemos
df = df.dropna(subset=['mes_ocorrencia']).copy()

print(f'Registros originais    : {len(df_raw)}')
print(f'Mantidos p/ modelagem  : {len(df)} ({(len(df))/len(df_raw):.1%})')

### 2.3 Engenharia de Features: Amostragem Negativa e Variáveis Temporais (Lags)

#### Por que Amostragem Negativa?

Os dados da SSP registram **apenas o que aconteceu**. Se o modelo treinasse só com esses dados, ele não saberia distinguir "bairro de alto risco" de "bairro com muitos registros por outras razões".

A solução é criar o **universo completo de cenários possíveis** — toda combinação de `bairro × mês × ano` — e marcar como `ocorreu = 0` os cenários sem ocorrência. O modelo aprende assim a fronteira entre risco e não-risco.

#### Features de Lag (Memória Temporal)

Introduzimos três variáveis históricas que dão "memória" ao modelo:

| Feature | Descrição | Por que ajuda |
|---|---|---|
| `lag_mensal` | Ocorrências no mês imediatamente anterior | Captura padrão de curto prazo: meses ruins tendem a ser seguidos de meses ruins |
| `lag_anual` | Ocorrências no mesmo mês do ano anterior | Captura sazonalidade: violência doméstica tem picos em determinadas épocas do ano |
| `media_movel_3m` | Média dos últimos 3 meses | Suaviza ruídos e representa tendência recente do bairro |

> **Atenção ao Data Leakage:** os lags são calculados com `.shift()`, ou seja, sempre usam informações do passado, nunca do período que estamos tentando prever. Isso evita que o modelo "veja o futuro" durante o treino.


In [ ]:
# Mapeamento do período e dia da semana
mapa_periodo = {
    'De Madrugada': 0, 'Pela Manhã': 1, 'A Tarde': 2, 'A Noite': 3,
    'Em Hora Incerta': -1, 'Não Informado': -1, 'Nao Informado': -1, 'nan': -1, 'None': -1
}
mapa_dia = {
    'Segunda-Feira': 0, 'Terca-Feira': 1, 'Terça-Feira': 1,
    'Quarta-Feira': 2, 'Quinta-Feira': 3,
    'Sexta-Feira': 4, 'Sabado': 5, 'Sábado': 5, 'Domingo': 6
}

# ── NOVO PARADIGMA: Previsão de Risco Espaço-Temporal com Amostragem Negativa ──
import itertools

# 1. Definimos o Universo de possibilidades
meses_possiveis = list(range(1, 13))
anos_possiveis = [2022, 2023, 2024, 2025, 2026]

# Bairros válidos (excluindo Desconhecido)
bairros_validos = df[df['bairro'] != 'Desconhecido']['bairro'].unique()

cenarios = list(itertools.product(
    bairros_validos, meses_possiveis, anos_possiveis
))
df_cenarios = pd.DataFrame(cenarios, columns=[
    'bairro', 'mes_ocorrencia', 'ano_ocorrencia'
])

# Recuperar o município de cada bairro
mapa_bairro_mun = df[df['bairro'] != 'Desconhecido'].groupby('bairro')['nome_municipio'].first().to_dict()
df_cenarios['nome_municipio'] = df_cenarios['bairro'].map(mapa_bairro_mun)

print(f'Universo de Cenários gerado: {len(df_cenarios)} combinações')

# 2. Agrupar as ocorrências reais
df_reais = df[df['bairro'] != 'Desconhecido'].copy()

agrupado = df_reais.groupby([
    'bairro', 'mes_ocorrencia', 'ano_ocorrencia'
]).size().reset_index(name='qtd_ocorrencias')
agrupado['ocorreu'] = 1 # Ocorreu pelo menos 1 crime

# 3. Left Join para cruzar o Universo com a Realidade
df_modelagem = pd.merge(
    df_cenarios, agrupado, 
    on=['bairro', 'mes_ocorrencia', 'ano_ocorrencia'], 
    how='left'
)
df_modelagem['ocorreu'] = df_modelagem['ocorreu'].fillna(0).astype(int)
df_modelagem['qtd_ocorrencias'] = df_modelagem['qtd_ocorrencias'].fillna(0).astype(int)

# --- Cálculo de Features Históricas (Lags) ---
# Ordenamos para garantir que o shift temporal funcione por bairro
df_modelagem = df_modelagem.sort_values(['bairro', 'ano_ocorrencia', 'mes_ocorrencia'])

# Lag 1: Ocorrências no mês anterior (no mesmo bairro)
df_modelagem['lag_mensal'] = df_modelagem.groupby('bairro')['qtd_ocorrencias'].shift(1).fillna(0)

# Lag 12: Ocorrências no mesmo mês do ano anterior (sazonalidade)
df_modelagem['lag_anual'] = df_modelagem.groupby('bairro')['qtd_ocorrencias'].shift(12).fillna(0)

# Média Móvel 3m: Média de crimes nos últimos 3 meses
df_modelagem['media_movel_3m'] = df_modelagem.groupby('bairro')['qtd_ocorrencias'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
).fillna(0)

# Remover cenários com mês inválido
df_modelagem = df_modelagem[df_modelagem['mes_ocorrencia'] > 0]

print(f"Total de registros na nova base de modelagem: {len(df_modelagem)}")
print(f"Cenários SEM ocorrência (0): {len(df_modelagem[df_modelagem['ocorreu'] == 0])}")
print(f"Cenários COM ocorrência (1): {len(df_modelagem[df_modelagem['ocorreu'] == 1])}")


### 2.4 Codificação das Variáveis Categóricas

O `LabelEncoder` transforma a coluna `nome_municipio` em valores numéricos (0 = Sorocaba, 1 = Votorantim ou vice-versa). 

A variável `bairro` **não** é codificada aqui por Label Encoding — ela receberá um **Target Encoding** na seção 4, calculado apenas sobre os dados de treino para evitar data leakage. Por isso, a lista `FEATURES` inclui `bairro_target_enc` como placeholder.


In [ ]:
# Encodings
le_municipio = LabelEncoder()
le_local     = LabelEncoder()

df_modelagem['municipio_enc'] = le_municipio.fit_transform(df_modelagem['nome_municipio'])

# Adicionamos local_ocorrencia como feature
# Primeiro recuperamos o local mais frequente de cada bairro-mês-ano
local_por_cenario = df_reais.groupby(
    ['bairro', 'mes_ocorrencia', 'ano_ocorrencia']
)['local_ocorrencia'].agg(lambda x: x.mode()[0] if len(x) > 0 else 'Residência').reset_index()

df_modelagem = pd.merge(df_modelagem, local_por_cenario,
                        on=['bairro', 'mes_ocorrencia', 'ano_ocorrencia'], how='left')
df_modelagem['local_ocorrencia'] = df_modelagem['local_ocorrencia'].fillna('Residência')
df_modelagem['local_enc'] = le_local.fit_transform(df_modelagem['local_ocorrencia'])

FEATURES = [
    'mes_ocorrencia',
    'bairro_target_enc',   # calculado no split
    'lag_mensal',
    'lag_anual',
    'media_movel_3m',
    'municipio_enc',
    'ano_ocorrencia',
]
TARGET = 'ocorreu'

print(f'Features: {FEATURES}')
print(f'Dataset preparado para Target Encoding no split.')

---
## 3. Análise Exploratória das Features

Antes de treinar qualquer modelo, visualizamos os dados para confirmar que as variáveis escolhidas realmente diferenciam hotspots de não-hotspots. Se uma variável não mostrasse diferença entre as classes, não haveria motivo para incluí-la.

### 3.1 Distribuição Histórica por Bairro

Identificamos os 15 bairros com mais ocorrências históricas e marcamos em vermelho os que estão acima do percentil 75 (nosso limiar de hotspot). Isso valida visualmente que existe heterogeneidade real entre bairros, requisito fundamental para um modelo preditivo funcionar.


In [ ]:
# Gráfico 1 — Distribuição por bairro

# Calculamos o Limiar histórico apenas para visualização no gráfico
contagem = df[df['bairro'] != 'Desconhecido'].groupby('bairro').size()
LIMIAR = contagem.quantile(0.75)
hotspot_bairros = set(contagem[contagem >= LIMIAR].index)

top15 = contagem.sort_values(ascending=False).head(15).reset_index(name='n')
fig, ax = plt.subplots(figsize=(11, 6))
cores = ['#c0392b' if b in hotspot_bairros else '#7f8c8d' for b in top15['bairro']]
ax.barh(top15['bairro'][::-1], top15['n'][::-1], color=cores[::-1])
ax.axvline(LIMIAR, color='black', linestyle='--', linewidth=1.2, label=f'Limiar P75 = {LIMIAR:.0f}')
ax.set_title('Top 15 Bairros com Mais Ocorrências Historicamente', fontsize=13, fontweight='bold', pad=10)
ax.set_xlabel('Volume Histórico')
ax.legend()
plt.tight_layout()
plt.show()


### 3.2 Sazonalidade: Período do Dia e Mês do Ano

Se período e mês não mostrassem variação, poderíamos removê-los do modelo. O gráfico abaixo confirma que há diferenças significativas, justificando sua inclusão como features.


In [ ]:
# Gráfico 2 — Variação por período e por mês
# Se essas variáveis não mostrassem diferença, não precisaríamos incluí-las no modelo.

meses_pt = {1:'Jan',2:'Fev',3:'Mar',4:'Abr',5:'Mai',6:'Jun',
            7:'Jul',8:'Ago',9:'Set',10:'Out',11:'Nov',12:'Dez'}

# Valores reais de periodo_ocorrencia no BigQuery (excluindo 'Não Informado' e 'Em Hora Incerta')
ordem_p   = ['De Madrugada', 'Pela Manhã', 'A Tarde', 'A Noite']
labels_p  = ['Madrugada',    'Manhã',      'Tarde',   'Noite']   # rótulo curto p/ exibição
presentes = [p for p in ordem_p if p in df['periodo_ocorrencia'].unique()]
labels_presentes = [labels_p[ordem_p.index(p)] for p in presentes]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

if presentes:
    por_p = (df[df['periodo_ocorrencia'].isin(presentes)]
             .groupby('periodo_ocorrencia', observed=True).size()
             .reindex(presentes)
             .fillna(0)
             .astype(int))
    por_p.index = labels_presentes
    por_p.plot(kind='bar', ax=axes[0], color='#2980b9', rot=0)
    axes[0].set_title('Ocorrências por Período do Dia')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Total')
    for p in axes[0].patches:
        h = p.get_height()
        if h > 0:
            axes[0].annotate(str(int(h)),
                             (p.get_x() + p.get_width()/2, h),
                             ha='center', va='bottom', fontsize=9)
else:
    axes[0].set_title('Período — sem dados reconhecíveis')
    axes[0].text(0.5, 0.5, str(df['periodo_ocorrencia'].unique()),
                 transform=axes[0].transAxes, ha='center', va='center', fontsize=8)

por_m = df.groupby('mes_ocorrencia').size()
por_m.index = por_m.index.map(meses_pt)
por_m.plot(kind='bar', ax=axes[1], color='#16a085', rot=0)
axes[1].set_title('Ocorrências por Mês (acumulado 2022–2026)')
axes[1].set_xlabel('')
axes[1].set_ylabel('Total')

plt.suptitle('Validação visual das features temporais', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../docs/features_temporais.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.3 Mapa de Calor: Interação entre Mês e Período

Este gráfico é especialmente importante porque revela **padrões de interação** entre mês e período do dia que não seriam visíveis isoladamente. Modelos como XGBoost conseguem aprender essas combinações automaticamente.


In [ ]:
# Gráfico 3 — Mapa de calor Mês x Período
# Este gráfico é importante porque mostra que a *combinação* das duas variáveis
# tem padrões que não aparecem quando olhamos cada uma isoladamente.
# Isso justifica que o modelo aprenda interações entre elas.

ordem_p   = ['De Madrugada', 'Pela Manhã', 'A Tarde', 'A Noite']
labels_p  = ['Madrugada',    'Manhã',      'Tarde',   'Noite']
presentes = [p for p in ordem_p if p in df['periodo_ocorrencia'].unique()]
labels_presentes = [labels_p[ordem_p.index(p)] for p in presentes]

if presentes:
    hm = (df[df['periodo_ocorrencia'].isin(presentes)]
          .groupby(['mes_ocorrencia', 'periodo_ocorrencia'], observed=True)
          .size()
          .unstack(fill_value=0)
          .reindex(columns=presentes, fill_value=0))
    hm.columns = labels_presentes
    hm.index   = hm.index.map(meses_pt)

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(hm, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.3, ax=ax,
                cbar_kws={'label': 'Nº de ocorrências'})
    ax.set_title('Mapa de Calor — Ocorrências por Mês e Período\n(Sorocaba e Votorantim, 2022–2026)')
    ax.set_xlabel('Período do Dia')
    ax.set_ylabel('Mês')
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.savefig('../docs/heatmap_mes_periodo.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Períodos encontrados nos dados (verifique grafia):')
    print(df['periodo_ocorrencia'].value_counts())


---
## 4. Divisão Treino/Teste e Balanceamento de Classes

### 4.1 Target Encoding para Bairros

Em vez de numerar os bairros arbitrariamente (Label Encoding), usamos **Target Encoding**: cada bairro é substituído pela sua taxa histórica de ocorrência de crime (`mean(ocorreu)` no treino). Assim, bairros de alto risco recebem valores próximos de 1 e bairros seguros recebem valores próximos de 0.

> **Por que calcular apenas no treino?** Se usarmos todos os dados (treino + teste) para calcular a média, o modelo "vê" informações do futuro durante o treino, isso é **data leakage** e infla artificialmente as métricas. O cálculo é feito exclusivamente no conjunto de treino e depois aplicado ao teste.

### 4.2 Divisão Estratificada (75% treino / 25% teste)

O parâmetro `stratify=y` garante que a proporção entre hotspot (1) e não-hotspot (0) seja a mesma nos dois conjuntos. Isso é essencial em datasets desbalanceados: sem estratificação, poderíamos ter por acaso todos os hotspots no treino e o teste ficaria sem exemplos positivos para avaliar.

### 4.3 Balanceamento com Random Undersampling

Como os cenários `ocorreu = 0` são muito mais frequentes que `ocorreu = 1`, o modelo tende a aprender "nunca preveja crime" e ainda assim ter alta acurácia. O `RandomUnderSampler` reduz a classe majoritária (sem crime) até igualar a minoritária (com crime), forçando o modelo a aprender os dois lados.

> **Por que Undersampling e não SMOTE?** O SMOTE gera exemplos sintéticos da classe minoritária, mas nossa classe minoritária já tem volume suficiente. O Undersampling é mais simples e evita ruído artificial nos dados.


In [ ]:
from imblearn.under_sampling import RandomUnderSampler

df_train, df_test = train_test_split(
    df_modelagem, test_size=0.25, random_state=42, stratify=df_modelagem['ocorreu']
)

# Target Encoding calculado APENAS no treino (sem leakage)
target_map = df_train.groupby('bairro')['ocorreu'].mean().to_dict()
df_train = df_train.copy()
df_test  = df_test.copy()
df_train['bairro_target_enc'] = df_train['bairro'].map(target_map).fillna(0)
df_test['bairro_target_enc']  = df_test['bairro'].map(target_map).fillna(0)

X_train = df_train[FEATURES]
y_train = df_train['ocorreu']
X_test  = df_test[FEATURES]
y_test  = df_test['ocorreu']

# MUDANÇA: ratio 1:3 em vez de 1:1 — preserva mais exemplos negativos
# O modelo aprende melhor a fronteira com mais contexto do que é "normal"
rus = RandomUnderSampler(sampling_strategy=0.33, random_state=42)
X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

print(f'Treino original : {len(X_train)} | Após Undersampling: {len(X_train_bal)}')
print(f'Distribuição após balanceamento:')
print(pd.Series(y_train_bal).value_counts())

# X e y para o CV global (com target encoding do treino)
X_cv = df_modelagem.copy()
X_cv['bairro_target_enc'] = X_cv['bairro'].map(target_map).fillna(0)
X = X_cv[FEATURES]
y = df_modelagem['ocorreu']

---
## 5. Função de Avaliação de Modelos

### Por que não usar apenas Acurácia?

Com dados desbalanceados, um modelo que sempre prevê "sem crime" pode ter 90% de acurácia, e ser completamente inútil. Por isso, priorizamos métricas mais informativas:

| Métrica | O que mede | Por que importa aqui |
|---|---|---|
| **Average Precision (AP CV-5)** | Área sob a curva Precisão-Recall em validação cruzada | Principal métrica de decisão, não depende de threshold fixo e é robusta ao desbalanceamento |
| **ROC-AUC** | Capacidade discriminatória geral | Quanto maior, melhor o modelo em separar as classes |
| **F1-Score** | Harmônica entre Precisão e Recall | Penaliza modelos que sacrificam uma das duas métricas |
| **Recall** | Proporção de hotspots reais identificados | Em segurança pública, é mais grave **perder** um hotspot real do que ter falsos alarmes |
| **Precisão** | Proporção de alertas que são reais | Alertas falsos em excesso esgotam recursos e credibilidade |

### Otimização de Threshold

O threshold padrão de 0.5 para classificação binária raramente é o ideal em dados desbalanceados. A função varre thresholds de 0.1 a 0.85 e escolhe o que maximiza o F1-Score no conjunto de teste.


In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score

resultados = {}

def avaliar_modelo(nome, modelo, X_tr, y_tr, X_te, y_te, mostrar_cm=True, mostrar_report=False):
    modelo.fit(X_tr, y_tr)
    
    auc = float('nan')
    best_thresh = 0.5
    if hasattr(modelo, 'predict_proba'):
        y_prob = modelo.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_te, y_prob)
        
        # Otimização de Threshold para F1
        best_f1 = 0
        for thresh in np.arange(0.1, 0.9, 0.05):
            pred_t = (y_prob >= thresh).astype(int)
            f1_t = f1_score(y_te, pred_t, zero_division=0)
            if f1_t > best_f1:
                best_f1 = f1_t
                best_thresh = thresh
        y_pred = (y_prob >= best_thresh).astype(int)
    else:
        y_pred = modelo.predict(X_te)

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, zero_division=0)

    # Validação Cruzada usando Average Precision (imune a threshold fixo)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_ap = cross_val_score(modelo, X, y, cv=cv, scoring='average_precision').mean()

    resultados[nome] = {
        'Acurácia':  round(acc, 4),
        'Precisão':  round(prec, 4),
        'Recall':    round(rec, 4),
        'F1-Score':  round(f1, 4),
        'AP CV-5':   round(cv_ap, 4), # Mudamos para AP (Average Precision)
        'ROC-AUC':   round(auc, 4),
    }

    print(f'Modelo: {nome}')
    print(f'  Threshold Otimizado : {best_thresh:.2f}')
    print(f'  F1-Score (Teste)    : {f1:.2%}')
    print(f'  AP CV-5 (Decisivo)  : {cv_ap:.2%}')
    print(f'  ROC-AUC             : {auc:.4f}')

    return modelo


---
## 6. Modelos de Referência (Baseline)

Antes dos modelos sofisticados, treinamos dois modelos simples que servem como **linha de base obrigatória**: se um modelo complexo não superar esses baselines, não há justificativa para usá-lo.

### 6.1 DummyClassifier — Predição Aleatória

Prevê classes de forma aleatória, respeitando a distribuição observada no treino. Qualquer modelo real deve superar essa referência com folga.


In [ ]:
dummy = DummyClassifier(strategy='stratified', random_state=42)
avaliar_modelo('DummyClassifier (referência aleatória)', dummy,
               X_train_bal, y_train_bal, X_test, y_test, mostrar_cm=False)

### 6.2 Regressão Logística — Baseline Linear

A Regressão Logística é o modelo mais simples que aprende relações lineares entre features e target. O parâmetro `class_weight='balanced'` compensa automaticamente o desbalanceamento sem precisar de undersampling.

Se a Regressão Logística já tiver bom desempenho, os dados têm separabilidade linear e modelos mais complexos podem não agregar muito. Se tiver desempenho fraco, significa que as relações são não-lineares, o que favorece árvores e XGBoost.


In [ ]:
# max_iter=1000 garante convergência — com múltiplas features o padrão (100) pode não ser suficiente
logreg = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
avaliar_modelo('Regressão Logística (baseline linear)', logreg,
               X_train_bal, y_train_bal, X_test, y_test)

---
## 7. Árvore de Decisão

A Árvore de Decisão é completamente **interpretável**: cada divisão pode ser lida como uma regra em linguagem natural (ex: *"Se o bairro tem score de risco > 0.3 E o lag mensal > 2, então → ALTO RISCO"*).

Testamos a árvore por dois motivos:

1. **Baseline para Random Forest:** a floresta nada mais é do que centenas de árvores em paralelo, se uma árvore já performa bem, sabemos que o ensemble vai melhorar ainda mais.
2. **Interpretabilidade:** em projetos de segurança pública, às vezes é necessário explicar a decisão do modelo, uma árvore permite isso.

O parâmetro `max_depth=6` limita o crescimento para evitar **overfitting** (memorização dos dados de treino sem generalizar para novos dados).


In [ ]:
dt = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42
)
dt_fit = avaliar_modelo('Árvore de Decisão', dt,
                        X_train_bal, y_train_bal, X_test, y_test, mostrar_report=True)

### 7.1 Visualização da Árvore

Exibimos apenas os 3 primeiros níveis (de 6 totais) para manter o gráfico legível. As folhas coloridas em azul indicam predição "Risco Normal" e as coloridas em laranja indicam "Alto Risco".


In [ ]:
# Visualizamos apenas os 3 primeiros níveis; com 6 níveis completos o gráfico fica ilegível
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt_fit, max_depth=3,
    feature_names=FEATURES,
    class_names=['Risco Normal', 'Alto Risco'],
    filled=True, rounded=True, fontsize=8, ax=ax
)
ax.set_title('Árvore de Decisão — primeiros 3 níveis (de 6 totais)')
plt.tight_layout()
plt.savefig('../docs/arvore_decisao.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. Random Forest

O Random Forest cria **300 árvores de decisão independentes**, cada uma treinada em uma amostra aleatória dos dados e com um subconjunto aleatório de features. A predição final é a votação majoritária entre todas as árvores.

**Vantagens em relação à árvore simples:**
- Muito menos sensível a overfitting (o ruído de uma árvore é cancelado pelas outras)
- Captura interações complexas entre variáveis
- Oferece estimativa robusta de importância de features

O parâmetro `max_features='sqrt'` faz cada árvore ver apenas `√n_features` variáveis, isso garante diversidade entre as árvores, que é o que torna o ensemble poderoso.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,        # usa todos os núcleos disponíveis
    random_state=42
)
rf_fit = avaliar_modelo('Random Forest', rf,
                        X_train_bal, y_train_bal, X_test, y_test, mostrar_report=True)

---
## 9. XGBoost com Busca de Hiperparâmetros (RandomizedSearchCV)

O XGBoost (Extreme Gradient Boosting) é o algoritmo principal. Diferente do Random Forest (árvores paralelas e independentes), o XGBoost constrói árvores **sequencialmente**: cada nova árvore foca nos erros da árvore anterior.

### Hiperparâmetros Buscados

| Parâmetro | Função |
|---|---|
| `n_estimators` | Número de árvores (mais = mais potente, mas mais lento) |
| `max_depth` | Profundidade máxima — controla complexidade e overfitting |
| `learning_rate` | Passo de aprendizado — menor = mais conservador e estável |
| `subsample` | Fração de amostras usadas por árvore — reduz overfitting |
| `colsample_bytree` | Fração de features por árvore — adiciona diversidade |
| `gamma` | Penalidade por criar novas divisões — regularização |
| `reg_alpha / reg_lambda` | Regularização L1 e L2 — controla complexidade dos pesos |

### Por que RandomizedSearchCV e não GridSearchCV?

O GridSearchCV testa **todas** as combinações possíveis, com 8 parâmetros e vários valores cada, seria inviável computacionalmente. O RandomizedSearchCV sorteia 20 combinações aleatórias e é estatisticamente eficiente para encontrar uma boa região do espaço de hiperparâmetros.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators':      [200, 400, 600, 800],
    'max_depth':         [3, 4, 5, 6, 7],
    'learning_rate':     [0.005, 0.01, 0.02, 0.05, 0.1],
    'subsample':         [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree':  [0.6, 0.7, 0.8, 0.9],
    'gamma':             [0, 0.1, 0.5, 1, 2, 5],
    'reg_alpha':         [0, 0.01, 0.1, 0.5, 1],
    'reg_lambda':        [0.5, 1, 2, 5],
    'min_child_weight':  [1, 3, 5, 10],   # NOVO: controla overfitting em folhas
}

# scale_pos_weight: diz ao XGBoost para dar mais peso à classe positiva
# Não precisamos de undersampling para o XGBoost — ele tem isso embutido
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos

xgb_base = XGBClassifier(
    scale_pos_weight=spw,
    eval_metric='aucpr',    # MUDANÇA: aucpr é mais adequado que logloss para dados desbalanceados
    use_label_encoder=False,
    random_state=42,
    tree_method='hist',     # mais rápido
)

print(f'scale_pos_weight = {spw:.2f}  (razão negativos/positivos no treino original)')
print('Iniciando busca (40 tentativas, ~3-5 min)...')

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=40,              # MUDANÇA: 40 em vez de 20 — explora mais o espaço
    scoring='average_precision',
    cv=5,                   # MUDANÇA: 5 folds em vez de 3 — estimativa mais estável
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# MUDANÇA: treina no conjunto original (sem undersampling)
# O scale_pos_weight já compensa o desbalanceamento internamente
xgb_search.fit(X_train, y_train)

print(f'\nMelhores parâmetros: {xgb_search.best_params_}')
print(f'Melhor AP (CV)     : {xgb_search.best_score_:.4f}')

xgb_best = xgb_search.best_estimator_
xgb_fit = avaliar_modelo('XGBoost', xgb_best, X_train, y_train, X_test, y_test, mostrar_report=True)

---
## 10. Comparação de Modelos

### Como interpretar o quadro comparativo?

A métrica decisiva é o **AP CV-5 (Average Precision em Validação Cruzada com 5 folds)**. Ela é preferida porque:
- **Não depende de threshold:** mede a performance em todos os thresholds possíveis
- **Validada em 5 folds:** avalia generalização, não apenas desempenho no teste fixo
- **Robusta ao desbalanceamento:** foca na classe positiva (hotspot), que é a que importa

Os modelos são ordenados por AP CV-5, o que estiver no topo é o vencedor.


In [ ]:
# Gerar quadro comparativo
df_res = pd.DataFrame(resultados).T.sort_values('AP CV-5', ascending=False)
styled_res = df_res.style.format('{:.2%}')     .highlight_max(subset=['AP CV-5', 'ROC-AUC'], color='lightgreen')     .set_caption("Quadro comparativo (ordenado por AP CV-5 — Métrica de Robustez)")

print("Quadro comparativo (ordenado por AP CV-5 — métrica decisiva):")
display(styled_res)


In [ ]:
cols  = ['Acurácia', 'Precisão', 'Recall', 'F1-Score', 'AP CV-5']
df_pl = df_res[cols].astype(float)

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(df_pl))
w = 0.15
cores = ['#2c3e50', '#2980b9', '#16a085', '#c0392b', '#8e44ad']

for i, (col, cor) in enumerate(zip(cols, cores)):
    ax.bar(x + (i - len(cols)/2)*w + w/2, df_pl[col], w, label=col, color=cor, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(df_pl.index, rotation=10, ha='right')
ax.set_ylim(0, 1.1)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.axhline(0.7, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='Limiar 70%')
ax.set_title('Comparação de Modelos — Métricas Principais')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('../docs/comparacao_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Importância das Variáveis (Feature Importance — XGBoost)

O XGBoost calcula a importância de cada feature com base em **quantas vezes ela foi usada para fazer uma divisão** e **quanto essa divisão reduziu o erro** em média. A feature mais importante é aquela que, quando o modelo precisa tomar uma decisão, é mais frequentemente a mais informativa.

Este gráfico é fundamental para:
- **Validar o modelo:** se `bairro_target_enc` (score de risco do bairro) não fosse importante, haveria um problema
- **Entender o fenômeno:** quais fatores realmente preveem violência doméstica
- **Orientar políticas:** se o lag mensal é importante, ações preventivas logo após picos podem ter maior impacto


In [ ]:
# Visualizar importância das variáveis do modelo vencedor (XGBoost)
import matplotlib.ticker as mtick

# Nomes amigáveis para o gráfico
nomes = {
    'mes_ocorrencia': 'Mês',
    'bairro_target_enc': 'Risco do Bairro (Target)',
    'lag_mensal': 'Crimes Mês Anterior',
    'lag_anual': 'Crimes Ano Anterior (Sazonalidade)',
    'media_movel_3m': 'Média Móvel (3 meses)',
    'municipio_enc': 'Município',
    'ano_ocorrencia': 'Ano'
}

# Pegamos as importâncias do XGBoost
fi = pd.Series(xgb_fit.feature_importances_, index=FEATURES).sort_values()
fi.index = fi.index.map(lambda x: nomes.get(x, x))

fig, ax = plt.subplots(figsize=(9, 5))
cores_fi = ['#c0392b' if v == fi.max() else '#7f8c8d' for v in fi.values]
ax.barh(fi.index, fi.values, color=cores_fi)

for i, (_, v) in enumerate(fi.items()):
    ax.text(v + 0.002, i, f'{v:.1%}', va='center', ha='left', fontsize=9)

ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Importância das Features — XGBoost\n(vermelho = variável de maior impacto preditivo)')
plt.tight_layout()
plt.savefig('../docs/importancia_features.png', dpi=150, bbox_inches='tight')
plt.show()

print('Ranking de importância (decrescente):')
print(fi.sort_values(ascending=False).apply('{:.2%}'.format).to_string())


---
## 12. Exemplo Prático de Predição

Para ilustrar o uso do modelo em produção, simulamos uma consulta real: **"Qual o risco de agressão no bairro X em março de 2026?"**

O código monta um vetor de features com:
- O score de risco histórico do bairro (Target Encoding)
- Os lags calculados a partir dos registros mais recentes daquele bairro
- Mês e ano do cenário de interesse

O threshold de 0.70 é mais conservador que o padrão (0.5), ou seja, o modelo só emite alerta "ALTO RISCO" se estiver com pelo menos 70% de confiança. Isso reduz falsos alarmes e aumenta a credibilidade dos alertas.


In [ ]:
# 1. Escolher o bairro mais frequente para o teste
bairro_top = df_modelagem.groupby('bairro')['qtd_ocorrencias'].sum().sort_values(ascending=False).index[0]

# 2. Buscar os valores de histórico (Lags) REAIS deste bairro no df_modelagem
info_bairro = df_modelagem[df_modelagem['bairro'] == bairro_top].sort_values(['ano_ocorrencia', 'mes_ocorrencia']).iloc[-1]

# Buscar o Score de Risco (Target Encoding) calculado no treino
b_score = target_map.get(bairro_top, 0)
m_idx = le_municipio.transform([info_bairro['nome_municipio']])[0]

# 3. Criar o cenário com as colunas que o modelo espera
cenario = pd.DataFrame([{
    'mes_ocorrencia': 3,    # Março
    'bairro_target_enc': b_score,
    'lag_mensal':     info_bairro['lag_mensal'],
    'lag_anual':      info_bairro['lag_anual'],
    'media_movel_3m': info_bairro['media_movel_3m'],
    'municipio_enc':  m_idx,
    'ano_ocorrencia': 2026
}])[FEATURES]

# 4. Predição com Threshold
prob = xgb_fit.predict_proba(cenario)[0]
pred = 1 if prob[1] >= 0.70 else 0

print(f'Cenário de Teste (Previsão para Março/2026):')
print(f'  Bairro      : {bairro_top}')
print(f'  Score de Risco do Bairro: {b_score:.4f}')
print(f'  Resultado   : {"ALTO RISCO" if pred == 1 else "RISCO NORMAL"}')
print(f'  Confiança   : {prob[1]:.2%}')


---
## 13. Exportação do Modelo e Artefatos

Salvamos todos os artefatos necessários para o aplicativo interativo (`app.py`) funcionar sem depender do notebook:

| Arquivo | Conteúdo | Para que serve no app |
|---|---|---|
| `modelo_hotspot_xgboost_v1.pkl` | Modelo XGBoost treinado | Fazer predições |
| `encoder_municipio.pkl` | LabelEncoder do município | Codificar a cidade selecionada pelo usuário |
| `target_encoding_bairros.json` | Score de risco de cada bairro | Converter nome do bairro em feature numérica |
| `stats_bairros.json` | Lags mais recentes por bairro | Preencher automaticamente as features históricas |
| `bairro_municipio_map.json` | Bairro → Município | Filtrar bairros por cidade na interface |


In [ ]:
# ── Exportar mapeamento bairro → município ──────────────────────────────────
# Necessário para o app.py filtrar bairros por cidade na interface interativa.
import json

bairro_municipio = (
    df.dropna(subset=["bairro", "nome_municipio"])
    .groupby("bairro")["nome_municipio"]
    .agg(lambda x: x.mode()[0])
    .to_dict()
)

with open("../models/bairro_municipio_map.json", "w", encoding="utf-8") as f:
    json.dump(bairro_municipio, f, ensure_ascii=False, indent=2)

print(f"Mapeamento salvo: {len(bairro_municipio)} bairros → models/bairro_municipio_map.json")
print(dict(list(bairro_municipio.items())[:5]))


In [ ]:
import joblib
import json

# Salvar o modelo final
fname = 'xgboost'
joblib.dump(xgb_fit, f'../models/modelo_hotspot_{fname}_v1.pkl')
joblib.dump(le_municipio, '../models/encoder_municipio.pkl')

# Exportar o mapeamento de Target Encoding para o App
with open('../models/target_encoding_bairros.json', 'w') as f:
    json.dump(target_map, f)

# Exportar estatísticas para o App
df_latest = df_modelagem.sort_values(['ano_ocorrencia', 'mes_ocorrencia']).groupby('bairro').last().reset_index()
stats_bairros = {}
for _, row in df_latest.iterrows():
    stats_bairros[row['bairro']] = {
        'lag_mensal': float(row['lag_mensal']),
        'lag_anual': float(row['lag_anual']),
        'media_movel_3m': float(row['media_movel_3m'])
    }

with open('../models/stats_bairros.json', 'w') as f:
    json.dump(stats_bairros, f)

print('Mapeamentos de Target Encoding e Stats salvos!')


In [ ]:
# RESUMO FINAL
vencedor = 'XGBoost'

print('RESUMO FINAL — Modelagem Preditiva com Amostragem Negativa (PI5)')
print('-' * 48)
print(f'Projeto BigQuery   : {NOME_DO_PROJETO}')
print(f'Tabela             : {ID_DA_TABELA}')
print(f'Registros originais: {len(df_raw)}')
print(f'Universo Modelagem : {len(df_modelagem)} cenários possíveis')
print(f'Bairros únicos     : {df_modelagem["bairro"].nunique()}')
print(f'Modelo selecionado : {vencedor}')
print()
print('Métricas no conjunto de teste:')
if vencedor in resultados:
    for k, v in resultados[vencedor].items():
        print(f'  {k:12s}: {v:.2%}')
else:
    print('Aviso: Resultados do XGBoost não encontrados no dicionário.')
print('-' * 48)
